In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display  # noqa: F401
from plotly.subplots import make_subplots

from src.preprocessing import clean_data, get_feature_lists, group_stores, load_raw_data

warnings.filterwarnings('ignore')
pio.templates.default = "plotly_white"
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

os.makedirs(".doc/notebook_plots/01_EDA", exist_ok=True)

# PART 1: EXPLORATORY DATA ANALYSIS (EDA) & PREPROCESSING

## Section 1: Data Overview & Missing Values Analysis

Understanding the shape, types, and completeness of the raw data is essential before any modeling. Missing values and type mismatches caught here prevent silent errors downstream.

In [2]:
df = load_raw_data()

display(Markdown(
    f"**Shape:** {df.shape[0]:,} rows x {df.shape[1]} columns"
))
display(Markdown("**Data types:**"))
display(df.dtypes.to_frame("Dtype"))
display(Markdown("**First 10 rows:**"))
display(df.head(10))

**Shape:** 150 rows x 12 columns

**Data types:**

,Dtype
Store,int64
Date,datetime64[us]
Weekly_Sales,float64
Holiday_Flag,Int64
Temperature,float64
Fuel_Price,float64
CPI,float64
Unemployment,float64
Year,float64
Month,float64


**First 10 rows:**

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,DayOfWeek
0,6,2011-02-18,1572117.54,<NA>,59.61,3.045,214.777523,6.858,2011.0,2.0,18.0,4.0
1,13,2011-03-25,1807545.43,0,42.38,3.435,128.616064,7.470,2011.0,3.0,25.0,4.0
2,17,2012-07-27,NaN,0,NaN,NaN,130.719581,5.936,2012.0,7.0,27.0,4.0
3,11,NaT,1244390.03,0,84.57,NaN,214.556497,7.346,NaN,NaN,NaN,NaN
4,6,2010-05-28,1644470.66,0,78.89,2.759,212.412888,7.092,2010.0,5.0,28.0,4.0
5,4,2010-05-28,1857533.70,0,NaN,2.756,126.160226,7.896,2010.0,5.0,28.0,4.0
6,15,2011-06-03,695396.19,0,69.80,4.069,134.855161,7.658,2011.0,6.0,3.0,4.0
7,20,2012-02-03,2203523.20,0,39.93,3.617,213.023622,6.961,2012.0,2.0,3.0,4.0
8,14,2010-12-10,2600519.26,0,30.54,3.109,NaN,NaN,2010.0,12.0,10.0,4.0
9,3,NaT,418925.47,0,60.12,3.555,224.132020,6.833,NaN,NaN,NaN,NaN


In [3]:
missing_df = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_%': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('Missing_%', ascending=False)

display(missing_df.style
    .format({"Missing_%": "{:.2f}%"})
    .set_caption("Missing Values Analysis"))

fig = go.Figure()
fig.add_trace(go.Bar(
    x=missing_df.index,
    y=missing_df['Missing_%'],
    text=missing_df['Missing_Count'],
    texttemplate='%{text} missing<br>(%{y:.1f}%)',
    textposition='outside',
    marker_color='steelblue'
))
fig.update_layout(
    title='Missing Values Distribution',
    xaxis_title='Column',
    yaxis_title='Missing Percentage (%)',
    height=400
)
fig.write_image(".doc/notebook_plots/01_EDA/Missing_Values_Distribution.png")
fig.show()

,Missing_Count,Missing_%
Date,18,12.00%
Month,18,12.00%
Day,18,12.00%
Temperature,18,12.00%
DayOfWeek,18,12.00%
Year,18,12.00%
Unemployment,15,10.00%
Weekly_Sales,14,9.33%
Fuel_Price,14,9.33%
Holiday_Flag,12,8.00%


### Expected output

![Missing_Values_Distribution](.doc/notebook_plots/01_EDA/Missing_Values_Distribution.png)

**Observation:** Missing values affect 11 of 12 columns, with rates between 8% and 12%. The pattern is widespread but uniform, making both drop and imputation viable strategies.

In [4]:
unique_data = {"Column": [], "Unique Count": [], "Sample Values": []}
for col in df.columns:
    unique_data["Column"].append(col)
    unique_data["Unique Count"].append(df[col].nunique())
    if col in ['Store', 'Holiday_Flag']:
        unique_data["Sample Values"].append(str(sorted([int(v) for v in df[col].dropna().unique()])))
    else:
        unique_data["Sample Values"].append("")

unique_df = pd.DataFrame(unique_data).set_index("Column")
display(unique_df.style.set_caption("Unique Values per Column"))

,Unique Count,Sample Values
Column,,
Store,20,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]"
Date,85,
Weekly_Sales,136,
Holiday_Flag,2,"[0, 1]"
Temperature,130,
Fuel_Price,120,
CPI,135,
Unemployment,104,
Year,3,


In [5]:
display(df.describe(include='all').round(2).style.set_caption("Statistical Summary"))

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day,DayOfWeek
count,150.000000,132,136.000000,138.000000,132.000000,136.000000,138.000000,135.000000,132.000000,132.000000,132.000000,132.000000
mean,9.870000,2011-05-07 09:05:27.272727,1249535.830000,0.080000,61.400000,3.320000,179.900000,7.600000,2010.860000,6.390000,16.520000,4.000000
min,1.000000,2010-02-05 00:00:00,268929.030000,0.000000,18.790000,2.510000,126.110000,5.140000,2010.000000,1.000000,1.000000,4.000000
25%,4.000000,2010-08-16 12:00:00,605075.720000,0.000000,45.590000,2.850000,131.970000,6.600000,2010.000000,4.000000,10.000000,4.000000
50%,9.000000,2011-05-09 12:00:00,1261423.860000,0.000000,62.980000,3.450000,197.910000,7.470000,2011.000000,6.000000,17.000000,4.000000
75%,15.750000,2012-01-14 18:00:00,1806386.200000,0.000000,76.340000,3.710000,214.930000,8.150000,2012.000000,9.000000,24.000000,4.000000
max,20.000000,2012-10-19 00:00:00,2771397.170000,1.000000,91.650000,4.190000,226.970000,14.310000,2012.000000,12.000000,31.000000,4.000000
std,6.230000,nan,647463.040000,0.270000,18.380000,0.480000,40.270000,1.580000,0.810000,3.210000,8.310000,0.000000


## Target Variable (Weekly Sales) Analysis

The target variable drives all modeling decisions. Examining its distribution reveals skewness, outliers, and missing values that determine preprocessing choices (e.g., log-transform, imputation strategy).

In [6]:
sales_clean = df['Weekly_Sales'].dropna()

sales_stats = pd.DataFrame({
    "Statistic": ["Count (non-null)", "Mean", "Median", "Std Dev", "Skewness", "Min", "Max", "Range"],
    "Value": [
        f"{len(sales_clean):,}",
        f"${sales_clean.mean():,.2f}",
        f"${sales_clean.median():,.2f}",
        f"${sales_clean.std():,.2f}",
        f"{sales_clean.skew():.2f}",
        f"${sales_clean.min():,.2f}",
        f"${sales_clean.max():,.2f}",
        f"${sales_clean.max() - sales_clean.min():,.2f}",
    ]
}).set_index("Statistic")

display(sales_stats.style.set_caption("Weekly Sales Statistics"))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Distribution (Histogram)', 'Outlier Detection (Boxplot)'),
    specs=[[{"type": "histogram"}, {"type": "box"}]]
)

mean_val = sales_clean.mean()
median_val = sales_clean.median()

fig.add_trace(
    go.Histogram(
        x=sales_clean,
        nbinsx=30,
        name='Weekly Sales',
        marker_color='steelblue',
        opacity=0.7
    ),
    row=1, col=1
)
fig.add_vline(x=mean_val, line_color="red", annotation_text="Mean", row=1, col=1)
fig.add_vline(x=median_val, line_dash="dash", line_color="green", annotation_text="Median", row=1, col=1)

fig.add_trace(
    go.Box(
        y=sales_clean,
        name='Weekly Sales',
        marker_color='steelblue',
        boxmean='sd',
        boxpoints="all",
        jitter=1,
        pointpos=0
    ),
    row=1, col=2
)

fig.update_layout(title_text='Weekly Sales Distribution', showlegend=False, height=450)
fig.update_xaxes(title_text='Weekly Sales ($)', row=1, col=1)
fig.update_yaxes(title_text='Frequency', row=1, col=1)
fig.update_yaxes(title_text='Weekly Sales ($)', row=1, col=2)
fig.write_image(".doc/notebook_plots/01_EDA/Weekly_Sales_Distribution.png")
fig.show()

display(Markdown(
    f"**Observation:** Weekly Sales distribution is approximately symmetric (skewness={sales_clean.skew():.2f}), "
    f"with mean (${mean_val:,.0f}) and median (${median_val:,.0f}) very close. "
    f"The boxplot shows the full range with no IQR-based outliers beyond the whiskers."
))

missing_target = df['Weekly_Sales'].isnull().sum()
missing_pct = missing_target / len(df) * 100
new_size = len(df) - missing_target

display(Markdown(f"""**Decision: Missing Weekly_Sales**
- Rows with missing target: **{missing_target}** ({missing_pct:.1f}%)
- Action: Drop these rows (never impute the target variable -- causes data leakage)
- Dataset size after drop: **{new_size:,}** rows"""))

,Value
Statistic,
Count (non-null),136
Mean,"$1,249,535.83"
Median,"$1,261,423.86"
Std Dev,"$647,463.04"
Skewness,0.08
Min,"$268,929.03"
Max,"$2,771,397.17"
Range,"$2,502,468.14"


**Observation:** Weekly Sales distribution is approximately symmetric (skewness=0.08), with mean ($1,249,536) and median ($1,261,424) very close. The boxplot shows the full range with no IQR-based outliers beyond the whiskers.

**Decision: Missing Weekly_Sales**
- Rows with missing target: **14** (9.3%)
- Action: Drop these rows (never impute the target variable -- causes data leakage)
- Dataset size after drop: **136** rows

### Expected output

![Weekly_Sales_Distribution](.doc/notebook_plots/01_EDA/Weekly_Sales_Distribution.png)

## Section 3: Categorical Variables Analysis (Store & Holiday_Flag)

Store identity and holiday periods are expected to be strong sales drivers. Analyzing their distributions and relationship with the target helps decide encoding strategy and whether store grouping is needed to avoid sparsity.

In [7]:
df_eda = df.dropna(subset=['Weekly_Sales']).copy()

display(Markdown("### Store Analysis"))

display(Markdown(
    f"- **Unique stores:** {df_eda['Store'].nunique()} | "
    f"**Missing:** {df_eda['Store'].isnull().sum()} | "
    f"**Values:** 1-20 (discrete category)"
))

store_stats = (
    df_eda.groupby('Store')['Weekly_Sales']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
    .rename(columns={'mean': 'Avg Weekly Sales', 'count': 'Observations'})
)
store_stats['Avg Weekly Sales'] = store_stats['Avg Weekly Sales'].map('${:,.0f}'.format)

display(Markdown("**Top 3 stores (by avg sales):**"))
display(store_stats.head(3))

display(Markdown("**Bottom 3 stores (by avg sales):**"))
display(store_stats.tail(3))

display(Markdown("### Holiday Flag Analysis"))

df_holiday = df_eda.dropna(subset=['Holiday_Flag']).copy()
missing_holiday = df_eda['Holiday_Flag'].isnull().sum()
missing_holiday_pct = missing_holiday / len(df_eda) * 100

display(Markdown(
    f"- **Unique values:** {df_holiday['Holiday_Flag'].nunique()} | "
    f"**Missing:** {missing_holiday} ({missing_holiday_pct:.1f}%) | "
    f"**Values:** 0 (non-holiday), 1 (holiday)"
))

holiday_stats = (
    df_holiday.groupby('Holiday_Flag')['Weekly_Sales']
    .agg(['mean', 'median', 'count'])
    .rename(columns={'mean': 'Mean Sales', 'median': 'Median Sales', 'count': 'Observations'})
)
holiday_stats.index = holiday_stats.index.map({0: 'Non-Holiday', 1: 'Holiday'})

display(holiday_stats.style
    .format({'Mean Sales': '${:,.0f}', 'Median Sales': '${:,.0f}', 'Observations': '{:,}'})
    .set_caption("Sales by Holiday Flag"))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Weekly Sales by Store', 'Weekly Sales by Holiday Flag'),
    specs=[[{"type": "box"}, {"type": "box"}]]
)

fig.add_trace(
    go.Box(x=df_eda['Store'], y=df_eda['Weekly_Sales'], marker_color='steelblue', name=''),
    row=1, col=1
)

fig.add_trace(
    go.Box(x=df_holiday['Holiday_Flag'].astype(str), y=df_holiday['Weekly_Sales'], marker_color='coral', name=''),
    row=1, col=2
)

fig.update_xaxes(title_text='Store ID', row=1, col=1)
fig.update_xaxes(title_text='Holiday (0=No, 1=Yes)', row=1, col=2)
fig.update_yaxes(title_text='Weekly Sales ($)', row=1, col=1)
fig.update_yaxes(title_text='Weekly Sales ($)', row=1, col=2)
fig.update_layout(title_text='Sales Distribution by Categorical Variables', showlegend=False, height=450)
fig.write_image(".doc/notebook_plots/01_EDA/Sales_by_Categorical_Variables.png")
fig.show()

store_sales_std = df_eda.groupby('Store')['Weekly_Sales'].std()
holiday_mean = df_holiday.groupby('Holiday_Flag')['Weekly_Sales'].mean()
holiday_diff_pct = (holiday_mean[1] - holiday_mean[0]) / holiday_mean[0] * 100

display(Markdown(
    f"**Observation:** Sales variance across stores is substantial (per-store std ranges from "
    f"${store_sales_std.min():,.0f} to ${store_sales_std.max():,.0f}), confirming that store identity "
    f"is a meaningful predictor. Holiday weeks show a {holiday_diff_pct:+.1f}% difference in mean sales "
    f"compared to non-holiday weeks, suggesting Holiday_Flag carries predictive signal."
))

display(Markdown(f"""**Decision: Missing Holiday_Flag**
- Rows with missing Holiday_Flag: **{df_eda['Holiday_Flag'].isnull().sum()}**
- Will be handled in Section 7 via clean_data() -- impute with mode (0) or drop"""))

### Store Analysis

- **Unique stores:** 20 | **Missing:** 0 | **Values:** 1-20 (discrete category)

**Top 3 stores (by avg sales):**

,Avg Weekly Sales,Observations
Store,,
4,"$2,173,759",6
14,"$2,092,878",9
13,"$1,997,235",9


**Bottom 3 stores (by avg sales):**

...

### Holiday Flag Analysis

- **Unique values:** 2 | **Missing:** 11 (8.1%) | **Values:** 0 (non-holiday), 1 (holiday)

,Mean Sales,Median Sales,Observations
Holiday_Flag,,,
Non-Holiday,"$1,239,576","$1,249,739",116
Holiday,"$1,333,024","$1,641,957",9


**Observation:** Sales variance across stores is substantial (per-store std ranges from $14,983 to $326,042), confirming that store identity is a meaningful predictor. Holiday weeks show a +7.5% difference in mean sales compared to non-holiday weeks, suggesting Holiday_Flag carries predictive signal.

**Decision: Missing Holiday_Flag**
- Rows with missing Holiday_Flag: **11**
- Will be handled in Section 7 via clean_data() -- impute with mode (0) or drop

### Expected output

![Sales_by_Categorical_Variables](.doc/notebook_plots/01_EDA/Sales_by_Categorical_Variables.png)

## Numeric Features Analysis and Outlier Detection

Outliers in economic indicators (CPI, Unemployment) or environmental features (Temperature) can distort model training. Detecting them with the 3-sigma rule quantifies how many rows would be lost and whether removal is justified.

In [8]:
numeric_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

display(Markdown("### Numeric Features - Distribution & Outlier Detection (+/-3sigma rule)"))

display(df[numeric_cols].describe().round(2).style.set_caption("Numeric Features Summary"))

outlier_bounds = {}
outlier_rows = []

for col in numeric_cols:
    data = df[col].dropna()
    mean = data.mean()
    std = data.std()
    lower = mean - 3 * std
    upper = mean + 3 * std

    outlier_bounds[col] = {'lower': lower, 'upper': upper, 'mean': mean, 'std': std}

    outliers_count = ((data < lower) | (data > upper)).sum()
    outliers_pct = (outliers_count / len(data)) * 100

    outlier_rows.append({
        'Feature': col,
        'mu-3sigma': f"{lower:.2f}",
        'mu+3sigma': f"{upper:.2f}",
        'Actual Min': f"{data.min():.2f}",
        'Actual Max': f"{data.max():.2f}",
        'Outliers': outliers_count,
        'Outliers %': f"{outliers_pct:.1f}%"
    })

outlier_df = pd.DataFrame(outlier_rows).set_index('Feature')
display(outlier_df.style.set_caption("Outlier Detection (+/-3sigma Rule)"))

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=numeric_cols,
    specs=[[{"type": "box"}, {"type": "box"}], [{"type": "box"}, {"type": "box"}]]
)

positions = [(1,1), (1,2), (2,1), (2,2)]

for col, (row, col_num) in zip(numeric_cols, positions):
    data = df[col].dropna()
    bounds = outlier_bounds[col]

    fig.add_trace(
        go.Box(y=data, name=col, marker_color='steelblue', boxmean='sd'),
        row=row, col=col_num
    )

    fig.add_hline(y=bounds['upper'], line_dash="dash", line_color="red",
                  annotation_text="mu+3sigma", row=row, col=col_num)
    fig.add_hline(y=bounds['lower'], line_dash="dash", line_color="red",
                  annotation_text="mu-3sigma", row=row, col=col_num)

fig.update_layout(title_text='Numeric Features Distribution (+/-3sigma Bounds)', showlegend=False, height=700)
fig.write_image(".doc/notebook_plots/01_EDA/Numeric_Features_Distribution.png")
fig.show()

display(Markdown(
    "**Observation:** Only Unemployment has outliers (5 cases, 3.7%)\n\n"
    "**Decision:** Remove rows with outliers in numeric features (minimal data loss, improves model quality)"
))

### Numeric Features - Distribution & Outlier Detection (+/-3sigma rule)

,Temperature,Fuel_Price,CPI,Unemployment
count,132.000000,136.000000,138.000000,135.000000
mean,61.400000,3.320000,179.900000,7.600000
std,18.380000,0.480000,40.270000,1.580000
min,18.790000,2.510000,126.110000,5.140000
25%,45.590000,2.850000,131.970000,6.600000
50%,62.980000,3.450000,197.910000,7.470000
75%,76.340000,3.710000,214.930000,8.150000
max,91.650000,4.190000,226.970000,14.310000


,mu-3sigma,mu+3sigma,Actual Min,Actual Max,Outliers,Outliers %
Feature,,,,,,
Temperature,6.26,116.53,18.79,91.65,0,0.0%
Fuel_Price,1.89,4.76,2.51,4.19,0,0.0%
CPI,59.07,300.72,126.11,226.97,0,0.0%
Unemployment,2.87,12.33,5.14,14.31,5,3.7%


**Observation:** Only Unemployment has outliers (5 cases, 3.7%)

**Decision:** Remove rows with outliers in numeric features (minimal data loss, improves model quality)

### Expected output

![Numeric_Features_Distribution](.doc/notebook_plots/01_EDA/Numeric_Features_Distribution.png)

## Date Feature Engineering and Temporal Analysis

Retail sales exhibit strong seasonality. Extracting temporal components (month, year) from the date column captures cyclical patterns that linear models cannot learn from a raw timestamp. Note: DayOfWeek is constant (all Fridays) and carries no predictive signal.

In [9]:
df_time = df.dropna(subset=['Date', 'Weekly_Sales']).copy()

display(Markdown("### Date Feature Engineering"))

dow_unique = sorted(df_time['DayOfWeek'].unique().astype(int))
dow_str = ", ".join(str(d) for d in dow_unique)
if len(dow_unique) == 1:
    dow_str = f"Only {dow_str} (Friday) -- constant, no predictive value"

date_info = pd.DataFrame({
    "Metric": [
        "Rows with valid dates",
        "Date range",
        "Time span (days)",
        "Years",
        "Months",
        "Days of week"
    ],
    "Value": [
        f"{len(df_time)}/{len(df)} ({len(df_time)/len(df)*100:.1f}%)",
        f"{df_time['Date'].min().date()} to {df_time['Date'].max().date()}",
        f"{(df_time['Date'].max() - df_time['Date'].min()).days}",
        str(sorted(df_time['Year'].unique().astype(int))),
        "1-12 (all present)",
        dow_str
    ]
}).set_index("Metric")

display(Markdown("Features created by `load_raw_data()`: Year, Month, Day, DayOfWeek"))
display(date_info.style.set_caption("Temporal Coverage"))

fig = px.line(
    df_time.sort_values('Date'),
    x='Date',
    y='Weekly_Sales',
    color='Store',
    title='Weekly Sales Over Time (by Store)',
    labels={'Weekly_Sales': 'Sales ($)'}
)
fig.update_layout(height=500, hovermode='x unified')
fig.write_image(".doc/notebook_plots/01_EDA/Weekly_Sales_Over_Time.png")
fig.show()

monthly_sales = df_time.groupby('Month')['Weekly_Sales'].agg(['mean', 'count'])

fig = go.Figure()
fig.add_trace(go.Bar(
    x=monthly_sales.index,
    y=monthly_sales['mean'],
    text=monthly_sales['count'],
    texttemplate='n=%{text}',
    textposition='outside',
    marker_color='steelblue'
))
fig.update_layout(
    title='Average Weekly Sales by Month (Seasonality)',
    xaxis_title='Month',
    yaxis_title='Average Weekly Sales ($)',
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    height=400
)
fig.write_image(".doc/notebook_plots/01_EDA/Average_Sales_by_Month.png")
fig.show()

display(Markdown(
    "**Observation:** Monthly sales show a strong December peak (holiday season). "
    "November is among the lowest months, likely due to small sample size. "
    "January-February sales are above the overall average.\n\n"
    "**Note:** These features will be included in model as numeric predictors"
))

### Date Feature Engineering

Features created by `load_raw_data()`: Year, Month, Day, DayOfWeek

,Value
Metric,
Rows with valid dates,118/150 (78.7%)
Date range,2010-02-05 to 2012-10-19
Time span (days),987
Years,"[np.int64(2010), np.int64(2011), np.int64(2012)]"
Months,1-12 (all present)
Days of week,"Only 4 (Friday) -- constant, no predictive value"


**Observation:** Monthly sales show a strong December peak (holiday season). November is among the lowest months, likely due to small sample size. January-February sales are above the overall average.

**Note:** These features will be included in model as numeric predictors

### Expected output

![Weekly_Sales_Over_Time](.doc/notebook_plots/01_EDA/Weekly_Sales_Over_Time.png)

![Average_Sales_by_Month](.doc/notebook_plots/01_EDA/Average_Sales_by_Month.png)

**TEMPORAL BIAS WARNING:**

Store comparisons may be biased if observations are not evenly distributed across seasons. This will be addressed in modeling (Part 2/3) via temporal features (Month, Holiday_Flag).

## Section 6: Correlation Analysis with Target Variable

Measuring linear relationships between numeric features and the target identifies which predictors carry signal. Checking inter-feature correlations also flags multicollinearity, which degrades interpretability of linear models.

In [10]:
df_corr = df.dropna(subset=['Weekly_Sales'])

numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

display(Markdown("### Correlation with Target Variable (Weekly_Sales)"))

correlations = {}
for col in numeric_features:
    data = df_corr[[col, 'Weekly_Sales']].dropna()
    corr = data[col].corr(data['Weekly_Sales'])
    correlations[col] = corr

corr_df = (
    pd.DataFrame([
        {
            'Feature': col,
            'Pearson r': corr,
            'Strength': 'Strong' if abs(corr) > 0.5 else 'Moderate' if abs(corr) > 0.3 else 'Weak',
            'Direction': 'negative' if corr < 0 else 'positive',
        }
        for col, corr in correlations.items()
    ])
    .sort_values('Pearson r', key=abs, ascending=False)
    .set_index('Feature')
)

display(corr_df.style
    .format({'Pearson r': '{:.3f}'})
    .set_caption("Pearson Correlation Coefficients with Weekly_Sales"))

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'{col}<br>r={correlations[col]:.3f}' for col in numeric_features]
)

positions = [(1,1), (1,2), (2,1), (2,2)]
for col, (row, col_num) in zip(numeric_features, positions):
    data = df_corr[[col, 'Weekly_Sales']].dropna()

    fig.add_trace(
        go.Scatter(
            x=data[col],
            y=data['Weekly_Sales'],
            mode='markers',
            marker=dict(color='steelblue', opacity=0.5, size=6),
            name=col
        ),
        row=row, col=col_num
    )

    z = np.polyfit(data[col], data['Weekly_Sales'], 1)
    p = np.poly1d(z)
    x_trend = np.linspace(data[col].min(), data[col].max(), 100)

    fig.add_trace(
        go.Scatter(
            x=x_trend,
            y=p(x_trend),
            mode='lines',
            line=dict(color='red', width=2),
            name='trend',
            showlegend=False
        ),
        row=row, col=col_num
    )

fig.update_layout(
    title_text='Numeric Features vs Weekly Sales (with Trend)',
    showlegend=False,
    height=600
)
fig.write_image(".doc/notebook_plots/01_EDA/Correlation_Scatter_Plots.png")
fig.show()

numeric_cols_all = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
corr_matrix = df_corr[numeric_cols_all].corr()

display(Markdown("### Full Correlation Matrix"))

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 10},
    colorbar=dict(title="Correlation")
))
fig.update_layout(
    title='Correlation Matrix - All Numeric Features',
    width=600,
    height=500
)
fig.write_image(".doc/notebook_plots/01_EDA/Correlation_Heatmap.png")
fig.show()

display(Markdown(
    "**Observation:** CPI has the strongest correlation with Weekly_Sales "
    "(r=-0.287, weak negative). All correlations are weak (|r| < 0.3), "
    "suggesting features interact in complex, non-linear ways.\n\n"
    "**Multicollinearity check:** No high inter-feature correlations (all |r| < 0.7) "
    "-- all features provide independent information."
))

### Correlation with Target Variable (Weekly_Sales)

,Pearson r,Strength,Direction
Feature,,,
CPI,-0.287,Weak,negative
Temperature,-0.166,Weak,negative
Unemployment,0.055,Weak,positive
Fuel_Price,-0.019,Weak,negative


### Full Correlation Matrix

**Observation:** CPI has the strongest correlation with Weekly_Sales (r=-0.287, weak negative). All correlations are weak (|r| < 0.3), suggesting features interact in complex, non-linear ways.

**Multicollinearity check:** No high inter-feature correlations (all |r| < 0.7) -- all features provide independent information.

### Expected output

![Correlation_Scatter_Plots](.doc/notebook_plots/01_EDA/Correlation_Scatter_Plots.png)

![Correlation_Heatmap](.doc/notebook_plots/01_EDA/Correlation_Heatmap.png)

## Missingness Analysis

Before choosing a cleaning strategy (drop vs. impute), we need to determine whether missing values follow a pattern (MNAR/MAR) or are random (MCAR). The mechanism dictates which imputation methods are valid.

In [11]:
display(Markdown("### Missingness Analysis"))

missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_info = (
    pd.DataFrame({
        'Missing Count': df.isnull().sum(),
        'Missing %': missing_pct,
    })
    .query('`Missing %` > 0')
    .sort_values('Missing %', ascending=False)
)

display(missing_info.style
    .format({'Missing %': '{:.2f}%'})
    .set_caption("Columns with Missing Values"))

fig = px.imshow(
    df.isnull().T,
    labels=dict(x="Row index", y="Column", color="Missing"),
    title="Missingness Heatmap (white = missing)",
    color_continuous_scale=["steelblue", "white"],
    aspect="auto",
)
fig.update_layout(height=350)
fig.write_image(".doc/notebook_plots/01_EDA/Missingness_Heatmap.png")
fig.show()

display(Markdown(
    "**Conclusion:** Missingness is mostly random across rows, but Date-derived features "
    "(Date, Year, Month, Day, DayOfWeek) and Temperature share identical missing counts "
    "(18 each), suggesting co-missingness in those rows. The pattern is closer to MAR "
    "than strict MCAR."
))

### Missingness Analysis

,Missing Count,Missing %
Date,18,12.00%
Day,18,12.00%
Temperature,18,12.00%
DayOfWeek,18,12.00%
Year,18,12.00%
Month,18,12.00%
Unemployment,15,10.00%
Fuel_Price,14,9.33%
Weekly_Sales,14,9.33%
Holiday_Flag,12,8.00%


**Conclusion:** Missingness is mostly random across rows, but Date-derived features (Date, Year, Month, Day, DayOfWeek) and Temperature share identical missing counts (18 each), suggesting co-missingness in those rows. The pattern is closer to MAR than strict MCAR.

### Expected output

![Missingness_Heatmap](.doc/notebook_plots/01_EDA/Missingness_Heatmap.png)

## Data Cleaning and Final Dataset

Applying the chosen cleaning strategy produces the final dataset for modeling. Comparing drop vs. impute retention rates ensures we make an informed trade-off between data quality and sample size.

In [12]:
display(Markdown("### Data Cleaning Execution (via `src.preprocessing`)"))

initial_rows = len(df)

df_drop = clean_data(df, strategy='drop')
df_impute = clean_data(df, strategy='impute')

strategy_df = pd.DataFrame({
    "Strategy": ["drop", "impute"],
    "Rows": [len(df_drop), len(df_impute)],
    "Retention %": [
        len(df_drop) / initial_rows * 100,
        len(df_impute) / initial_rows * 100,
    ],
}).set_index("Strategy")

display(Markdown(f"**Initial dataset:** {initial_rows} rows"))
display(strategy_df.style
    .format({"Retention %": "{:.1f}%", "Rows": "{:,}"})
    .set_caption("Cleaning Strategy Comparison"))

# Use drop strategy as required by exam instructions
df_clean = df_drop.copy()

display(Markdown(
    f"**Decision:** Using **drop** strategy as required by exam instructions "
    f"({len(df_drop)} rows retained)"
))

df_clean = group_stores(df_clean)

store_group_df = pd.DataFrame({
    "Metric": ["Original stores", "Grouped stores", "Samples in group 999"],
    "Value": [
        df_clean['Store'].nunique(),
        df_clean['Store_Grouped'].nunique(),
        int((df_clean['Store_Grouped'] == 999).sum())
        if 999 in df_clean['Store_Grouped'].values else 0,
    ],
}).set_index("Metric")

display(store_group_df.style.set_caption("Store Grouping Summary"))

cat_features, num_features = get_feature_lists(df_clean)

display(Markdown(
    f"**Feature lists:**\n"
    f"- Categorical: `{cat_features}`\n"
    f"- Numeric: `{num_features}`"
))

display(Markdown("---\n### Final Dataset Summary"))

summary_cols = [
    'Store', 'Store_Grouped', 'Weekly_Sales',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
]

display(Markdown(
    f"**Shape:** {df_clean.shape[0]} rows x {df_clean.shape[1]} columns | "
    f"**Retention:** {len(df_clean) / initial_rows * 100:.1f}% of original data"
))

display(Markdown(f"**Columns:** `{list(df_clean.columns)}`"))

missing_final = df_clean.isnull().sum()
display(Markdown(
    f"**Missing values:** {missing_final.sum()} total "
    f"({'none -- dataset is clean' if missing_final.sum() == 0 else 'see details below'})"
))

display(df_clean[summary_cols].describe().round(2).style
    .format("${:,.2f}", subset=["Weekly_Sales"])
    .set_caption("Final Dataset Statistics"))

df_clean.to_csv('src/input/Walmart_clean.csv', index=False)

display(Markdown(
    f"Cleaned dataset saved to `src/input/Walmart_clean.csv` ({len(df_clean)} rows). "
    f"Data is ready for modeling (Parts 2 and 3)."
))

### Data Cleaning Execution (via `src.preprocessing`)

**Initial dataset:** 150 rows

,Rows,Retention %
Strategy,,
drop,71,47.3%
impute,113,75.3%


**Decision:** Using **drop** strategy as required by exam instructions (71 rows retained)

,Value
Metric,
Original stores,19
Grouped stores,6
Samples in group 999,39


...

**Shape:** 71 rows x 13 columns | **Retention:** 47.3% of original data

**Columns:** `['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Day', 'DayOfWeek', 'Store_Grouped']`

**Missing values:** 0 total (none -- dataset is clean)

,Store,Store_Grouped,Weekly_Sales,Temperature,Fuel_Price,CPI,Unemployment
count,71.000000,71.000000,$71.00,71.000000,71.000000,71.000000,71.000000
mean,9.990000,552.920000,"$1,211,317.93",59.710000,3.310000,178.790000,7.310000
std,6.260000,495.980000,"$692,410.23",17.160000,0.490000,39.550000,0.950000
min,1.000000,3.000000,"$268,929.03",18.790000,2.550000,126.140000,5.140000
25%,4.000000,7.000000,"$526,281.54",45.020000,2.820000,132.560000,6.540000
50%,8.000000,999.000000,"$1,166,117.85",60.710000,3.440000,196.920000,7.350000
75%,16.000000,999.000000,"$1,827,488.19",75.040000,3.740000,214.830000,8.090000
max,20.000000,999.000000,"$2,771,397.17",89.420000,4.170000,226.970000,9.340000


Cleaned dataset saved to `src/input/Walmart_clean.csv` (71 rows). Data is ready for modeling (Parts 2 and 3).

## DATASET LIMITATIONS & CONSTRAINTS

Documenting these limitations upfront is essential: it informs the jury about known constraints that bound model performance, and it manages expectations by making explicit what the data can and cannot support.

### Critical Issues

**1. Sample-to-Feature Ratio (Overfitting Risk)**
- Final dataset: 71 samples (drop strategy)
- After one-hot encoding: ~13 features (7 numeric + 5 store dummies + 1 holiday dummy with drop_first)
- Ratio: ~5.5:1 (71 samples / 13 features), recommended: >=10:1
- Impact: Structural overfitting likely, regularization essential

**2. Class Imbalance**
- Holiday_Flag: 9 holidays vs 116 non-holidays (pre-cleaning counts)
- Impact: Model may underfit holiday patterns due to limited examples

**3. Store Representation**
- Several stores with < 5 samples (grouped into Store_Grouped=999)
- Impact: Limited learning capacity for under-represented stores

**4. Data Loss**
- Original dataset: 150 rows
- After cleaning (drop): ~71 rows (~47% retention)
- Drop strategy removes all rows with any missing value

### Implications for Modeling

- Test set size will be small (~21 samples with 30% split)
- Cross-validation recommended for robust evaluation
- Regularization (Ridge/Lasso) mandatory to combat overfitting
- Feature reduction strategies may help (store grouping implemented)
- Results should be interpreted with caution given limited data

### Recommendations

- Use cross-validation instead of single train-test split
- Apply regularization (Ridge/Lasso) in Parts 2 & 3
- Consider temporal split (chronological) to avoid data leakage
- Document model uncertainty and confidence intervals
- Collect more data if possible for production deployment